# **Preparación de los Datos**

**Extracción del Archivo Tratado**

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("datos_tratados.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

**Eliminación de Columnas Irrelevantes**


In [ ]:
df = df.drop(columns=["customerID", "Churn"])

In [ ]:
df.head()

In [ ]:
df.columns

customerID fue eliminada porque es un identificador único y no aporta información útil para predecir la evasión de clientes.

Churn fue eliminada porque ya se creó la variable churn_binario, que representa la variable objetivo en formato numérico, más adecuada para modelos predictivos.



In [ ]:
df["account.Charges.Total"] = pd.to_numeric(df["account.Charges.Total"], errors="coerce")

In [ ]:
df.info()

account.Charges.Total debería ser numérica (float)

**Encoding**

In [ ]:
df["account.Charges.Total"] = df["account.Charges.Total"].fillna(df["account.Charges.Total"].median())

In [ ]:
df.info()

account.Charges.Total mostraba 7032 datos pero el dataset tiene 7043 filas. Significa que habia 11 valores nulos. Antes de entrenar modelos normalmente hay que correguir los valores NaN.

In [ ]:
X = df.drop("churn_binario", axis=1)
y = df["churn_binario"]

In [ ]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [ ]:
X_encoded.head()

In [ ]:
X_encoded.shape

In [ ]:
X_encoded.info()

**Verificación de la Proporción de Cancelación (Churn)**

In [ ]:
conteo_churn = df["churn_binario"].value_counts()
conteo_churn

5174 clientes no cancelaron

1869 clientes cancelaron

In [ ]:
proporcion_churn = df["churn_binario"].value_counts(normalize=True)
proporcion_churn

73% de clientes permanecen, 27% cancelan. Esto es desbalance moderado, no extremo.  Teniendo en cuenta que muchos modelos funcionan bien sin corregirlo; en este reto aun no se corregirá.

**Normalización o Estandarización (si es necesario)**

Se realizara la normalizacion mas adelante, se escalara después de dividir los datos para evitar generar un data leakage.

# **Correlación y Selección de Variables**


**Análisis de Correlación**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# seleccionar variables numéricas + target
df_corr = df[[
    "customer.tenure",
    "account.Charges.Monthly",
    "account.Charges.Total",
    "churn_binario"
]]

# matriz de correlación
corr_matrix = df_corr.corr()

corr_matrix

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm")
plt.title("Matriz de correlación")
plt.show()

Se analizó la matriz de correlación entre las variables numéricas del dataset y la variable objetivo churn_binario. Los resultados muestran que customer.tenure presenta una correlación negativa moderada con la cancelación, lo que sugiere que los clientes con mayor antigüedad tienen menor probabilidad de cancelar el servicio.

Por otro lado, account.Charges.Monthly muestra una correlación positiva débil, indicando que los clientes con cargos mensuales más altos presentan una ligera tendencia a cancelar.

Y TotalCharges vs churn muestra una correlación negativa débil, es decir, clientes que han pagado más dinero acumulado tienden a quedarse. Lo que significa que la antigüedad del cliente es un factor importante en la cancelación.

**Análisis Dirigido**

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x="churn_binario", y="customer.tenure", data=df)

plt.title("Tiempo como cliente vs Cancelación")
plt.xlabel("Cancelación (0 = No, 1 = Sí)")
plt.ylabel("Meses como cliente")

plt.show()

El análisis mediante boxplot muestra que los clientes que cancelaron el servicio generalmente tienen un menor tiempo de permanencia en la empresa. Esto sugiere que la cancelación ocurre con mayor frecuencia entre clientes relativamente nuevos, mientras que aquellos con mayor antigüedad tienden a permanecer en el servicio.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x="churn_binario", y="account.Charges.Total", data=df)

plt.title("Gasto total vs Cancelación")
plt.xlabel("Cancelación (0 = No, 1 = Sí)")
plt.ylabel("Gasto total")

plt.show()

El análisis del gasto total mediante un boxplot muestra que los clientes que cancelaron el servicio presentan, en promedio, valores más bajos de gasto acumulado en comparación con aquellos que permanecen en la empresa. Esto sugiere que los clientes que cancelan tienden a hacerlo en etapas relativamente tempranas de su relación con el servicio, mientras que los clientes con mayor gasto total suelen ser aquellos con mayor permanencia y fidelidad.

# **Modelado Predictivo**

**Separación de Datos**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

Para evaluar adecuadamente el desempeño del modelo, el conjunto de datos fue dividido en datos de entrenamiento y prueba utilizando la función train_test_split de Scikit-learn. Se utilizó una proporción de 80% para entrenamiento y 20% para prueba. Además, se empleó el parámetro stratify para mantener la misma proporción de la variable objetivo en ambos conjuntos, garantizando una evaluación más representativa del modelo.

**Creación de Modelos**

In [ ]:
columnas_numericas = [
    "customer.tenure",
    "account.Charges.Monthly",
    "account.Charges.Total"
]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[columnas_numericas] = scaler.fit_transform(X_train[columnas_numericas])
X_test_scaled[columnas_numericas] = scaler.transform(X_test[columnas_numericas])

Modelo 1 (requiere normalización) Regresión Logística.

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo_lr = LogisticRegression(max_iter=1000)

modelo_lr.fit(X_train_scaled, y_train)

In [ ]:
y_pred_lr = modelo_lr.predict(X_test_scaled)

Modelo 2 (NO requiere normalización) Random Forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(random_state=42)

modelo_rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = modelo_rf.predict(X_test)

Para predecir la cancelación de clientes se implementaron dos modelos de aprendizaje automático: Regresión Logística y Random Forest.

La Regresión Logística es un modelo sensible a la escala de las variables, ya que optimiza sus coeficientes mediante métodos iterativos y puede verse afectado si las variables tienen magnitudes muy diferentes. Por esta razón, las variables numéricas fueron estandarizadas utilizando StandardScaler, lo que permite que tengan media 0 y desviación estándar 1.

Por otro lado, el modelo Random Forest está basado en árboles de decisión, los cuales realizan divisiones basadas en umbrales de las variables y no dependen de la escala de los datos. Por ello, este modelo se entrenó utilizando los datos sin normalización.

Esta estrategia permite comparar el desempeño de un modelo sensible a la escala de los datos con otro que no requiere este tipo de preprocesamiento.


**Evaluación de los Modelos**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

Evaluación de los Modelos

In [ ]:
print("Regresión Logística")

print("Accuracy:", accuracy_score(y_test, y_pred_lr))

print(confusion_matrix(y_test, y_pred_lr))

print(classification_report(y_test, y_pred_lr))

In [ ]:
print("Random Forest")

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print(confusion_matrix(y_test, y_pred_rf))

print(classification_report(y_test, y_pred_rf))

¿Cuál modelo tuvo mejor desempeño?

La Regresión Logística tuvo un desempeño mejor. Ambos modelos presentaron una precisión global similar (79%). Sin embargo, la Regresión Logística mostró un mejor desempeño en la detección de clientes que cancelan el servicio, con un mayor recall y F1-score para la clase de cancelación, generaliza mejor y no presenta overfitting.

¿Hay overfitting o underfitting?

In [ ]:
from sklearn.metrics import accuracy_score

y_train_pred_lr = modelo_lr.predict(X_train_scaled)

print("Train accuracy:", accuracy_score(y_train, y_train_pred_lr))
print("Test accuracy:", accuracy_score(y_test, y_pred_lr))

La Regresión Logística mostró un desempeño consistente entre el conjunto de entrenamiento (81%) y el conjunto de prueba (79%). La pequeña diferencia entre ambos valores indica que el modelo generaliza adecuadamente y no presenta señales significativas de overfitting o underfitting.

In [ ]:
y_train_pred_rf = modelo_rf.predict(X_train)

print("Train accuracy:", accuracy_score(y_train, y_train_pred_rf))
print("Test accuracy:", accuracy_score(y_test, y_pred_rf))

0.997 - 0.79 ≈ 0.20. Caso de OVERFITTING. El modelo aprendió casi perfectamente el dataset de entrenamiento pero pierde capacidad al predecir datos nuevos, por eso el train accuracy es casi 100%. Esto ocurre porque Random Forest crea muchos árboles, árboles muy profundos, dataset relativamente pequeño, muchas variables categóricas.

Cómo se podría mejorar

Ajustando hiperparámetros para reducir la complejidad del modelo

` RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    random_state=42
) `

# **Interpretación y Conclusiones**




**Análisis de la Importancia de las Variables**

Importancia en Regresión Logística



In [ ]:
import pandas as pd

coeficientes = pd.DataFrame({
    "Variable": X_train_scaled.columns,
    "Coeficiente": modelo_lr.coef_[0]
})

coeficientes = coeficientes.sort_values(by="Coeficiente", ascending=False)

coeficientes.head(10)

Importancia en Random Forest

In [ ]:
import pandas as pd

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo_rf.feature_importances_
})

importancias = importancias.sort_values(by="Importancia", ascending=False)

importancias.head(10)

El análisis de importancia de variables permitió identificar los factores más influyentes en la cancelación de clientes. Tanto la Regresión Logística como Random Forest coinciden en que variables relacionadas con los cargos del servicio y el tipo de servicio de internet tienen un impacto significativo en la probabilidad de churn. En particular, el servicio de fibra óptica, los cargos totales y mensuales, y el método de pago electrónico aparecen como factores relevantes. Asimismo, el tiempo que el cliente lleva con la empresa (tenure) también muestra una fuerte influencia, indicando que los clientes más nuevos presentan una mayor probabilidad de cancelar el servicio.

# **Conclusión**

El presente análisis tuvo como objetivo identificar los factores que influyen en la cancelación de clientes y desarrollar modelos predictivos capaces de anticipar este comportamiento. Para ello, se aplicaron técnicas de análisis exploratorio de datos, normalización de variables, entrenamiento de modelos de machine learning y evaluación de desempeño.

Se evaluaron dos modelos de clasificación: Regresión Logística y Random Forest, con el fin de comparar su capacidad para predecir la cancelación de clientes.


Evaluación del rendimiento de los modelos

Ambos modelos obtuvieron una exactitud cercana al 79% en el conjunto de prueba, lo que indica un desempeño general similar en la predicción de cancelación de clientes. Sin embargo, al analizar el rendimiento entre los datos de entrenamiento y prueba se observaron diferencias importantes:

Regresión Logística: Accuracy entrenamiento: 0.81. Accuracy prueba: 0.79. La pequeña diferencia entre ambos valores indica que el modelo generaliza adecuadamente y no presenta señales significativas de overfitting, lo que lo convierte en un modelo estable para este problema.

Random Forest: Accuracy entrenamiento: 0.997. Accuracy prueba: 0.79. En este caso se observa una diferencia considerable entre entrenamiento y prueba, lo que indica overfitting. El modelo aprendió demasiado bien los datos de entrenamiento, perdiendo capacidad para generalizar a nuevos datos.

Debido a esto, la Regresión Logística se considera el modelo más equilibrado para este análisis, ya que presenta un mejor balance entre capacidad predictiva y generalización.

Factores que más influyen en la cancelación

El análisis de importancia de variables permitió identificar los factores que tienen mayor impacto en la probabilidad de cancelación. Tanto la Regresión Logística como Random Forest coinciden en la relevancia de varias variables clave.

Variables más influyentes identificadas:

1. Cargos totales del cliente (Total Charges): Esta variable fue la más importante en el modelo Random Forest. Clientes con mayores gastos acumulados muestran patrones de comportamiento que pueden influir en la decisión de cancelar el servicio.

2. Tiempo como cliente (Tenure): El tiempo que un cliente lleva con la empresa es uno de los factores más importantes. Los clientes con menos tiempo en la empresa tienen mayor probabilidad de cancelar, lo que indica que los primeros meses de relación con el cliente son críticos para su fidelización.

3. Cargos mensuales (Monthly Charges): Los clientes con pagos mensuales más altos presentan mayor probabilidad de churn. Esto puede indicar que los clientes perciben que el servicio es costoso o que existen alternativas más económicas en el mercado.

4. Tipo de servicio de internet (Fiber Optic): El modelo de Regresión Logística mostró que los clientes con servicio de fibra óptica presentan mayor probabilidad de cancelación. Esto podría estar relacionado con expectativas más altas sobre la calidad del servicio o con el costo del mismo.

5. Método de pago (Electronic Check): Los clientes que utilizan electronic check como método de pago presentan mayor tendencia a cancelar el servicio. Esto podría indicar menor compromiso o menor automatización en el proceso de pago.

Principales factores asociados al churn

Con base en el análisis, los factores más relacionados con la cancelación son: Altos cargos mensuales, Alto gasto total acumulado, Bajo tiempo de permanencia como cliente, Uso de internet de fibra óptica, Método de pago electrónico (electronic check).

Estos factores permiten identificar patrones importantes en el comportamiento de los clientes.

Estrategias de retención recomendadas

Con base en los resultados obtenidos, se pueden proponer las siguientes estrategias para reducir la cancelación de clientes:

1. Programas de fidelización para clientes nuevos: Dado que los clientes con menor tiempo de permanencia presentan mayor probabilidad de cancelación, se recomienda implementar: beneficios durante los primeros meses, descuentos de bienvenida y seguimiento personalizado al inicio del servicio.

2. Revisión de planes con cargos mensuales elevados: Los clientes con pagos mensuales más altos presentan mayor riesgo de churn. Se recomienda: ofrecer paquetes alternativos más económicos, promociones para clientes con altos cargos.

3. Mejora en la experiencia del servicio de fibra óptica: Dado que los usuarios de fibra óptica presentan mayor probabilidad de cancelación, es importante: evaluar la calidad del servicio, mejorar soporte técnico, gestionar expectativas del cliente.

4. Incentivar métodos de pago automáticos: El uso de electronic check se relaciona con mayor churn. Se podrían implementar: descuentos por pago automático, incentivos para métodos de pago más estables.


Conclusión general

El análisis de churn permitió identificar patrones relevantes en el comportamiento de los clientes y desarrollar modelos predictivos capaces de anticipar la cancelación. La Regresión Logística demostró ser el modelo más adecuado debido a su capacidad de generalización y estabilidad en el rendimiento.

Además, el análisis de importancia de variables permitió identificar factores clave relacionados con los costos del servicio, el tiempo de permanencia del cliente y características del servicio contratado.

Estos hallazgos pueden ser utilizados por la empresa para diseñar estrategias de retención más efectivas, enfocándose especialmente en clientes nuevos, clientes con altos cargos mensuales y usuarios de ciertos servicios específicos.

In [ ]:
# Contenido del README
readme_text = """
# Predicción de Cancelación de Clientes (Churn) - TelecomX

## Descripción del Proyecto
La cancelación de clientes (churn) es uno de los principales problemas para las empresas de telecomunicaciones, ya que impacta directamente en los ingresos y la estabilidad del negocio. El objetivo de este proyecto es analizar el comportamiento de los clientes de TelecomX y desarrollar modelos de Machine Learning capaces de predecir si un cliente cancelará el servicio. Además, se identifican las variables más influyentes en la cancelación para proponer estrategias de retención. Este análisis permite a la empresa comprender mejor a sus clientes y tomar decisiones basadas en datos para reducir la tasa de cancelación.

El dataset utilizado proviene de:
datos_tratados.csv

## Estructura del Proyecto
- notebook.ipynb
- graficos/   # Carpeta con gráficos generados
- README.md

## Uso del Proyecto
1. Abrir el notebook.ipynb en Google Colab.
2. Instalar librerías necesarias: pandas, matplotlib, seaborn.
3. Ejecutar todas las celdas para generar gráficos y análisis.

## Autor
Karen Muñoz
karenchamorro27@gmail.com
"""

# Guardar el README.md
with open("README.md", "w") as f:
    f.write(readme_text)